5G用户预测

import relevant libraties

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from visualization import roc

Load data

In [26]:
data = pd.read_csv('train.csv')
x = data.drop(['id', 'target'], axis=1)
y = data['target']

Split the data set

In [27]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

Characteristic standardization（Logical regression）

In [28]:
scaler = StandardScaler()
x_train_lr = scaler.fit_transform(x_train)
x_test_lr = scaler.transform(x_test)

Logical regression

In [29]:
# Model construction
model = LogisticRegression(
        max_iter=1000,
)
model.fit(x_train_lr, y_train)
y_pred_lr = model.predict_proba(x_test_lr)[:,1]
# Model test
auc_lr = roc_auc_score(y_test, y_pred_lr)
print("logical regression AUC:" , auc_lr);

logical regression AUC: 0.8350520635474818


Random forest

In [30]:
# Model construction
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    n_jobs= -1 # 多线程
)
model.fit(x_train, y_train)
# Model test
y_pred_rf = model.predict_proba(x_test)[:,1]
auc_rf = roc_auc_score(y_test, y_pred_rf)
print("random forest AUC:" , auc_rf);

random forest AUC: 0.9107644293800162


lightGBM

In [31]:
# Model construction
model = lgb.LGBMClassifier(
    num_iterations=500,
    learning_rate=0.05,
    num_leaves=64,
)
model.fit(x_train, y_train)
# 模型测试（使用测试集)
y_pred_lgb = model.predict_proba(x_test)[:,1]
auc_lgb = roc_auc_score(y_test, y_pred_lgb)
print("lightGBM AUC:" , auc_lgb);

[LightGBM] [Info] Number of positive: 8447, number of negative: 631553
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.081666 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9416
[LightGBM] [Info] Number of data points in the train set: 640000, number of used features: 58
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.013198 -> initscore=-4.314371
[LightGBM] [Info] Start training from score -4.314371
lightGBM AUC: 0.9232922144698781


集成学习

In [32]:
# Logical regression、Random forest、LightGBM
total_auc_1 = auc_lr + auc_rf + auc_lgb
weight_lr = auc_lr / total_auc_1
weight_rf = auc_rf / total_auc_1
weight_lgb = auc_lgb / total_auc_1
y_pred_1 = weight_lr * y_pred_lr + weight_rf * y_pred_rf + weight_lgb * y_pred_lgb
auc_1 = roc_auc_score(y_test, y_pred_1)
print("Weighted integrationLogical regression、Random forest、LightGBM AUC:" , auc_1);
# Random forest、LightGBM
total_auc_2 = auc_rf + auc_lgb
weight_rf = auc_rf / total_auc_2
weight_lgb = auc_lgb / total_auc_2
y_pred_2 = weight_rf * y_pred_rf + weight_lgb * y_pred_lgb
auc_2 = roc_auc_score(y_test, y_pred_2)
print("Weighted integrationRandom forest、LightGBM AUC:" , auc_2);

加权集成逻辑回归、随机森林、LightGBM AUC: 0.9172580622299797
加权集成随机森林、LightGBM AUC: 0.9230857318544169


Visualization

In [33]:
roc(y_test, {'logical regression':y_pred_lr,'random forest':y_pred_rf, 'lightGBM':y_pred_lgb}, 'roc.png')

ROC曲线保存成功： roc.png
